In [ ]:
# ===============================
# Core Python Libraries
# ===============================
import os
import pickle
import joblib
import numpy as np
import pandas as pd
from collections import Counter
from scipy.special import expit  # sigmoid

# ===============================
# Data Visualization
# ===============================
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from PIL import Image, ImageTk
import cv2
# ===============================
# System & Utilities
# ===============================
import os
import joblib
import numpy as np
from tqdm import tqdm
import re

# ===============================
# NLP: NLTK 
# ===============================
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
from nltk.util import ngrams

In [ ]:
# ===============================
# Scikit-learn: Preprocessing, Models, Metrics
# ===============================
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
import joblib, os
from sklearn.linear_model import SGDClassifier, PassiveAggressiveClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from imodels import GreedyTreeClassifier
from sklearn.model_selection import train_test_split
# ===============================
# TensorFlow / Keras
# ===============================
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model, load_model
from tensorflow.keras.layers import Dense, Input, Embedding, Conv1D, GlobalMaxPooling1D, LSTM, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

# ===============================
# Transformers (Hugging Face)
# ===============================
import torch
# ===============================
# Custom Modules
# ===============================
from metrics_calculator import MetricsCalculator
from graphs import GraphPlotter
from transformers import (
    AutoTokenizer, AutoModel,
    RobertaTokenizer, RobertaModel,
    BertTokenizer, BertForSequenceClassification,
    XLNetTokenizer, XLNetForSequenceClassification
)
# ===============================
# Model Directory
# ===============================

In [ ]:
# NLTK Downloads
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
from tqdm import tqdm
tqdm.pandas()

In [ ]:
MODEL_DIR = "model_execution_success"
os.makedirs(MODEL_DIR, exist_ok=True)

In [ ]:
from pathlib import Path
import pandas as pd

def resolve_dataset_path():
    candidates = [
        Path("Dataset/benchmark_results.csv"),
        Path("../Dataset/benchmark_results.csv"),
        Path("major_project/Dataset/benchmark_results.csv"),
        Path("../major_project/Dataset/benchmark_results.csv"),
    ]
    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        "Dataset/benchmark_results.csv was not found. "
        "Run this notebook from the repository root or Advanced_BERT folder."
    )

def upload_dataset():
    global file_path, df
    file_path = str(resolve_dataset_path())
    df = pd.read_csv(file_path)
    if "Execution_Success" not in df.columns:
        raise ValueError("Dataset must contain the Execution_Success target column.")
    print(f"Loaded dataset: {file_path}")
    print(f"Dataset shape: {df.shape}")
    print("Input columns:", [c for c in df.columns if c != "Execution_Success"])
    print("Target distribution:")
    print(df["Execution_Success"].value_counts(dropna=False))


In [ ]:
upload_dataset()

In [ ]:
df

In [ ]:
target_cols = ["Execution_Success"]

In [ ]:
def preprocess_data(df, save_path=None, target_cols=None):
    """Use every dataset column as input except Execution_Success."""
    global label_encoders
    label_encoders = {}

    target_cols = target_cols or ["Execution_Success"]
    missing = [c for c in target_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing target columns: {missing}")

    # Isolate target before constructing model inputs.
    target_df = df[target_cols].copy()
    input_df = df.drop(columns=target_cols).copy()
    if input_df.empty:
        raise ValueError("No input columns remain after excluding the target.")

    # Normalize Execution_Success to stable TRUE/FALSE labels.
    for col in target_cols:
        if col == "Execution_Success":
            mapping = {
                "true": "TRUE", "1": "TRUE", "1.0": "TRUE", "yes": "TRUE",
                "false": "FALSE", "0": "FALSE", "0.0": "FALSE", "no": "FALSE"
            }
            target_df[col] = target_df[col].astype(str).str.strip().str.lower().map(mapping)
        else:
            target_df[col] = target_df[col].astype("string").str.strip()

    valid = target_df.notna().all(axis=1) & input_df.notna().any(axis=1)
    dropped = int((~valid).sum())
    input_df = input_df.loc[valid].reset_index(drop=True)
    target_df = target_df.loc[valid].reset_index(drop=True)
    if input_df.empty:
        raise ValueError("No valid rows remain after removing missing targets/empty inputs.")

    # Preserve all feature fields (including SQL, status, metrics, and NL_Query).
    # Prefix each value with its source column name before BERT embedding.
    input_parts = []
    for col in input_df.columns:
        values = input_df[col].fillna("").astype(str).str.strip()
        input_parts.append(col + ": " + values)

    X_text = pd.concat(input_parts, axis=1).agg(" | ".join, axis=1)
    X = X_text.str.replace(r"\\s+", " ", regex=True).str.strip().tolist()

    Y_dict = {}
    for col in target_cols:
        encoder = LabelEncoder()
        encoded = encoder.fit_transform(target_df[col].astype(str))
        if col == "Execution_Success" and set(encoder.classes_) != {"FALSE", "TRUE"}:
            raise ValueError(
                "Execution_Success must contain both TRUE and FALSE; "
                f"found {list(encoder.classes_)}"
            )
        label_encoders[col] = encoder
        Y_dict[col] = encoded

    if save_path:
        os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
        cleaned = input_df.copy()
        for col in target_cols:
            cleaned[col] = target_df[col]
        cleaned.to_csv(save_path, index=False)

    print("All input columns:", list(input_df.columns))
    print("Target:", target_cols)
    print(f"Prepared rows: {len(X):,}; dropped rows: {dropped:,}")
    for col in target_cols:
        print("Label mapping:", col, {
            label: int(label_encoders[col].transform([label])[0])
            for label in label_encoders[col].classes_
        })
        print(target_df[col].value_counts().to_string())

    return X, Y_dict


In [ ]:
def plot_target_distributions(Y_dict):

    # Convert to DataFrame
    y_df = pd.DataFrame(Y_dict)

    # Loop through each target column
    for col in y_df.columns:

        # Print total number of rows
        total_rows = len(y_df[col])
        print(f"{col} → Total Rows: {total_rows}")

        # Plot count distribution
        plt.figure(figsize=(6, 4))
        ax = sns.countplot(x=y_df[col])

        # Add count labels on each bar
        for p in ax.patches:
            height = p.get_height()
            ax.annotate(
                f'{height}',                     # text
                (p.get_x() + p.get_width()/2, height),  # position
                ha='center', va='bottom', 
                fontsize=10, fontweight='bold'
            )

        plt.title(f'Class Distribution: {col}')
        plt.xlabel('Class')
        plt.ylabel('Count')
        plt.tight_layout()
        plt.show()

In [ ]:
def Preprocess_Dataset_button():
    global df, X, Y_dict
    global metrics_calculator_dict, target_cols

    X, Y_dict = preprocess_data(
        df,
        save_path=f"{MODEL_DIR}/cleaned_data.csv",
        target_cols=target_cols
    )

    metrics_calculator_dict = {}

    print("Extracted Label Classes:\n")

    for col, le in label_encoders.items():

        labels = list(le.classes_)  # class names list

        # Print readable summary
        print(f"Target: {col}")
        for idx, class_name in enumerate(labels):
            print(f"  {idx}: {class_name}")
        print()

        # Save labels
        label_file = os.path.join(MODEL_DIR, f"labels_{col}.npy")
        np.save(label_file, np.array(labels), allow_pickle=True)

        # Create metrics calculator
        metrics_calculator_dict[col] = MetricsCalculator(
            labels,
            text_widget=None
        )

    print("Labels saved successfully!\n")

    # Plot distributions

In [ ]:
Preprocess_Dataset_button()

In [ ]:
plot_target_distributions(Y_dict)

In [ ]:
def eda_nlp_analysis():
    global df,X, Y_dict

    X_text = X
    num_words=100
    top_n_words=20
    
    print("Generating NLP EDA Visualizations..."+"\n\n")

    # Flatten all tokens from all texts
    all_tokens = [word for doc in X_text for word in word_tokenize(doc)]

    # --- 1. WordCloud ---
    word_freq = Counter(all_tokens)
    wc = WordCloud(width=800, height=400, max_words=num_words, background_color='white').generate_from_frequencies(word_freq)

    plt.figure(figsize=(10, 5))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    plt.title(f"Top {num_words} Words - WordCloud")
    plt.show()

    # --- 2. Top-N Frequent Words ---
    common_words = word_freq.most_common(top_n_words)
    words, counts = zip(*common_words)
    plt.figure(figsize=(10, 5))
    sns.barplot(x=list(counts), y=list(words), palette="viridis")
    plt.title(f"Top {top_n_words} Most Frequent Words")
    plt.xlabel("Count")
    plt.ylabel("Word")
    plt.show()

    # --- 3. Document Length Histogram ---
    doc_lengths = [len(word_tokenize(doc)) for doc in X_text]
    plt.figure(figsize=(10, 5))
    sns.histplot(doc_lengths, bins=20, kde=True, color='teal')
    plt.title("Distribution of Document Lengths (in words)")
    plt.xlabel("Number of Words per Document")
    plt.ylabel("Frequency")
    plt.show()

    # --- 4. POS Tag Frequency ---
    all_pos = [tag for _, tag in pos_tag(all_tokens)]
    pos_counts = Counter(all_pos).most_common()
    pos_tags, pos_freqs = zip(*pos_counts)
    plt.figure(figsize=(10, 5))
    sns.barplot(x=list(pos_tags), y=list(pos_freqs), palette="coolwarm")
    plt.title("Part of Speech (POS) Tag Frequency")
    plt.xlabel("POS Tag")
    plt.ylabel("Frequency")
    plt.xticks(rotation=45)
    plt.show()

    # --- 5. Bigram Frequency Plot ---
    bigrams = list(ngrams(all_tokens, 2))
    bigram_freq = Counter(bigrams).most_common(top_n_words)
    bigram_labels = [' '.join(b) for b, _ in bigram_freq]
    bigram_counts = [count for _, count in bigram_freq]

    plt.figure(figsize=(10, 5))
    sns.barplot(x=bigram_counts, y=bigram_labels, palette="magma")
    plt.title(f"Top {top_n_words} Bigrams")
    plt.xlabel("Count")
    plt.ylabel("Bigram")
    plt.show()



In [ ]:
import warnings
warnings.filterwarnings("ignore")

eda_nlp_analysis()

In [ ]:
def Transformer_feature_extraction(
        texts,
        model_name,
        batch_size=32,
        pooling='cls',  # or 'mean'
        device=None):

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    
    model.eval()
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    all_embs = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Extracting Trasformer embeddings"):
        batch = texts[i:i+batch_size]
        enc = tokenizer(batch, padding=True, truncation=True, return_tensors='pt')
        enc = {k: v.to(device) for k, v in enc.items()}

        with torch.no_grad():
            output = model(**enc)

        # output.last_hidden_state: sequence of hidden states for each character’s embedding after upsampling
        # output.pooler_output: the pooled [CLS]-style output (for classification / sentence-level features)
        if pooling == 'cls':
            embs = output.pooler_output
        elif pooling == 'mean':
            # mean over sequence dimension
            embs = output.last_hidden_state.mean(dim=1)
        else:
            raise ValueError("pooling must be 'cls' or 'mean'")

        all_embs.append(embs.cpu().numpy())

    X = np.vstack(all_embs)
    return X, model

In [ ]:
def feature_extraction(X_text, method='Transfoermer_Embeddings', model_dir='model', is_train=True):
    x_file = os.path.join(model_dir, f'X_{method}.pkl')

    print( f"[INFO] Feature extraction method: {method}, Train mode: {is_train}"+"\n\n")
    model_name='sentence-transformers/all-MiniLM-L6-v2'

    if is_train:
        if os.path.exists(x_file):
            print(f"[INFO] Loading cached embeddings from {x_file}"+"\n\n")
            X = joblib.load(x_file)
        else:
            print( "[INFO] Computing embeddings..."+"\n\n")
            X, model = Transformer_feature_extraction(X_text, model_name=model_name, pooling='mean')
            os.makedirs(model_dir, exist_ok=True)
            joblib.dump(X, x_file)
    else:
        print( "[INFO] Performing embedding extraction for testing..."+"\n\n")
        X, model = Transformer_feature_extraction(X_text, model_name=model_name, pooling='mean')

    return X

In [ ]:
def feature_extraction_button():
    global X, features
    global features_dict, labels_dict, target_cols

    from imblearn.over_sampling import SMOTE, RandomOverSampler
    # ---------------- FEATURE EXTRACTION ----------------
    features = feature_extraction( X, method='Transfoermer_Embeddings', model_dir=MODEL_DIR, is_train=True)

    original_features = {}
    original_labels = {}

    for i, (key, y) in enumerate(Y_dict.items(), start=1):

        print(f"\n🔹 Processing '{key}' (Original)...\n")

        original_features[f'features{i}'] = features
        original_labels[f'Y{i}'] = y

        print(
            f"Dataset: features{i}.shape = {features.shape}, "
            f"Y{i}.shape = {y.shape}\n"
        )

    # ---------------- BALANCING ----------------
    resampled_features = {}
    resampled_labels = {}

    target_names = target_cols

    for idx, name in enumerate(target_names, start=1):

        X_target = original_features[f'features{idx}']
        y_target = original_labels[f'Y{idx}']

        # -------- Before Sampling --------
        classes_before, counts_before = np.unique(
            y_target,
            return_counts=True
        )

        min_class_count = np.min(counts_before)

        try:

            # Use RandomOverSampler when any class has < 6 samples
            if min_class_count < 6:

                sampler = RandomOverSampler(
                    random_state=42
                )

                sampler_name = "RandomOverSampler"

            else:

                sampler = SMOTE(
                    random_state=42
                )

                sampler_name = "SMOTE"

            X_res, y_res = sampler.fit_resample(
                X_target,
                y_target
            )

        except Exception as e:

            print(
                f"Sampling Failed for {name}: {str(e)}\n"
                f"Original data retained.\n"
            )

            X_res = X_target
            y_res = y_target
            sampler_name = "None"

        # -------- After Sampling --------
        classes_after, counts_after = np.unique(
            y_res,
            return_counts=True
        )

        resampled_features[name] = X_res
        resampled_labels[name] = y_res

        print(f"{sampler_name} Applied for {name}:")

        print(f"Before: {dict(zip(classes_before, counts_before))}")

        print(f"After : {dict(zip(classes_after, counts_after))}")

        print( f"New shape: {X_res.shape}, {y_res.shape}\n")

        # -------- Bar Graph --------
        plt.figure(figsize=(8, 5))

        x_before = np.arange(len(classes_before))
        x_after = np.arange(len(classes_after))

        plt.bar(
            x_before - 0.2,
            counts_before,
            width=0.4,
            label="Before Sampling"
        )

        plt.bar(
            x_after + 0.2,
            counts_after,
            width=0.4,
            label="After Sampling"
        )

        plt.xticks(x_before, classes_before)
        plt.xlabel("Class Labels")
        plt.ylabel("Number of Samples")
        plt.title(
            f"{name}: Before vs After {sampler_name}"
        )
        plt.legend()
        plt.tight_layout()
        plt.show()

    # ---------------- SAVE RESULTS ----------------
    features_dict = resampled_features
    labels_dict = resampled_labels

    print("\nFeature Extraction + Class Balancing Completed Successfully\n")

    # Optional Summary
    print("Balanced Dataset Summary:\n")

    for key in labels_dict:

        classes, counts = np.unique(
            labels_dict[key],
            return_counts=True
        )
        print( f"{key}: {dict(zip(classes, counts))}")

In [ ]:
feature_extraction_button()

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from imodels import BoostedRulesClassifier
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier

from imodels import BoostedRulesClassifier, FIGSClassifier

def train_ml_models(Algorithm_prefix, features_dict, Y_dict, algorithm):
    ml_models = {}

    model_mapping = {
        "LR": LogisticRegression,
        "ETC": ExtraTreesClassifier,
        "DTC": DecisionTreeClassifier,
        "BRC": BoostedRulesClassifier,
        "Proposed": RandomForestClassifier
    }

    if algorithm not in model_mapping:
        raise ValueError(f"Unknown algorithm: {algorithm}")

    for target_name, y_encoded in Y_dict.items():

        X = features_dict[target_name]
        model_cls = model_mapping[algorithm]

        model_path = f"{MODEL_DIR}/Transformer_Embeddings_{target_name}_{algorithm}_model.pkl"
        algo_name = f"{Algorithm_prefix} {algorithm} [{target_name}]"

        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y_encoded,
            test_size=0.2,
            random_state=42,
            stratify=y_encoded
        )

        # ---------------- MODEL INITIALIZATION ----------------
        if algorithm == "LR":
            mdl = model_cls(max_iter=50, C=0.1, solver='liblinear')

        elif algorithm == "ETC":
            mdl = model_cls(n_estimators=10, max_depth=5, min_samples_split=10, min_samples_leaf=5)

        elif algorithm == "DTC":
            mdl = model_cls(max_depth=3, min_samples_split=10, min_samples_leaf=5)

        elif algorithm == "BRC":
            mdl = model_cls()

        elif algorithm == "Proposed":
            mdl = model_cls()

        # ---------------- LOAD OR TRAIN ----------------
        if os.path.exists(model_path):
            print(f"[LOAD] Existing {algorithm} model for: {target_name}")
            mdl = joblib.load(model_path)
        else:
            print(f"[TRAIN] {algorithm} model → Target: {target_name}")
            mdl.fit(X_train, y_train)
            joblib.dump(mdl, model_path, compress=("lzma", 3))

        # ---------------- PREDICTION ----------------
        y_pred = mdl.predict(X_test)

        try:
            y_score = mdl.predict_proba(X_test)
        except:
            y_score = None

        # ---------------- METRICS ----------------
        if target_name in metrics_calculator_dict:
            metrics_calculator_dict[target_name].calculate_metrics(
                algo_name,
                y_pred,
                y_test,
                y_score
            )

        ml_models[f"{target_name}_{algorithm}"] = mdl

    return ml_models

In [ ]:
def existing_classifier1():
   
    global features_dict,labels_dict
    models = train_ml_models("Transformer_Embeddings", features_dict, labels_dict, "LR")

In [ ]:
import warnings
warnings.filterwarnings("ignore")
existing_classifier1()

In [ ]:
def existing_classifier2():

    global features_dict,labels_dict
    models = train_ml_models("Transformer_Embeddings", features_dict, labels_dict, "ETC")
existing_classifier2()

In [ ]:
def Proposed_classifier():

    global features_dict,labels_dict
    models = train_ml_models("Transformer_Embeddings", features_dict, labels_dict, "Proposed")
Proposed_classifier()

In [ ]:
def graph():

    for target_name, calculator in metrics_calculator_dict.items():

        if calculator.metrics_df.empty:
            print(f"No metrics found for {target_name}\n")
            continue

        # Copy main table
        df_h = calculator.metrics_df[['Algorithm', 'Accuracy', 'Precision', 'Recall', 'F1-Score']].copy()
        df_h = df_h.round(3)

        first_alg = df_h['Algorithm'].iloc[0]
        detected_name = re.search(r'\[(.*?)\]', first_alg)

        if detected_name:
            extracted_target = detected_name.group(1)
        else:
            extracted_target = target_name

        print(f"\n=== Performance for {extracted_target} ===\n")

        df_h['Algorithm'] = df_h['Algorithm'].str.replace(
            r'\s*\[.*?\]', '', regex=True
        )

        print(df_h.to_string(index=False))
        print("\n")

In [ ]:
graph()

In [ ]:
def Prediction():

    import re
    import joblib
    import numpy as np
    import pandas as pd
    from pathlib import Path

    global df_result

    # ---------------- LOAD TEST DATA ----------------
    test_filename = r"C:\Users\ADVAITHA-AI\Desktop\Cyber insurance code\Dataset\Test.csv"

    df_test1 = pd.read_csv(test_filename)
    df_result = df_test1.copy()

    print(f"\nLoaded Test File: {test_filename}")
    print(f"Total Test Samples: {len(df_test1)}")

    # ---------------- PREPROCESS ----------------
    print("\nPreprocessing test data...")
    df_test, _ = preprocess_data(df_test1)

    # ---------------- FEATURE EXTRACTION ----------------
    print("\nExtracting Transformer embeddings...")

    base_features_test = feature_extraction(
        df_test,
        method='Transformer_Embeddings',
        is_train=False
    )

    # ---------------- MODEL DIRECTORY ----------------
    model_dir = Path(MODEL_DIR)

    if not model_dir.exists():
        print(f"\nModel directory not found: {model_dir}")
        return

    # ---------------- FIND MODELS ----------------
    model_files = list(
        model_dir.glob(
            "Transformer_Embeddings_*_Proposed_model.pkl"
        )
    )

    if len(model_files) == 0:
        print("\nNo proposed models found.")
        return

    print(f"\nFound {len(model_files)} proposed models")

    # ---------------- PREDICTION ----------------
    for model_file in model_files:

        match = re.search(
            r"Transformer_Embeddings_(.*?)_Proposed_model\.pkl",
            model_file.name
        )

        if not match:
            continue

        target_name = match.group(1).strip()

        print("\n" + "=" * 60)
        print(f"Loading Proposed Model : {target_name}")
        print("=" * 60)

        label_file = model_dir / f"labels_{target_name}.npy"

        if not label_file.exists():
            print(f"Labels file not found: {label_file}")
            continue

        try:
            labels = np.load(
                label_file,
                allow_pickle=True
            )

            model = joblib.load(model_file)

            y_pred = model.predict(base_features_test)

            mapped = [labels[int(i)] for i in y_pred]

            pred_col = f"Predicted_{target_name}"

            df_result[pred_col] = mapped

            print(f"Prediction completed for {target_name}")

        except Exception as e:
            print(f"Error in {target_name}: {str(e)}")

    # ---------------- DISPLAY RESULTS ----------------
    pred_cols = [
        c for c in df_result.columns
        if c.startswith("Predicted_")
    ]

    print("\n")
    print("=" * 80)
    print("PREDICTION RESULTS")
    print("=" * 80)

    for i, row in df_result.iterrows():

        print(f"\nRow {i + 1}")
        print("-" * 50)

        for col in df_result.columns:

            if col not in pred_cols:
                print(f"{col}: {row[col]}")

        print("\nPredictions:")
        for col in pred_cols:
            print(f"{col}: {row[col]}")

    print("\nPrediction completed successfully.")

    return df_result

In [ ]:
Prediction()

In [ ]:
df_result